# Prediccion Temprana de Riesgo de Retraso en Proyectos Software

**Tipo de modelo:** Clasificacion binaria (RandomForestClassifier)
**Variable objetivo:** retraso_proyecto (1 = retraso, 0 = en plazo)

**Lo que aprenderas:**
1. Como detectar senales de riesgo antes de que el retraso sea irreversible
2. Por que el coste de un falso negativo es mayor que el de un falso positivo aqui
3. Como ajustar el umbral segun el coste del error
4. Feature importance: que metricas operativas predicen mejor el retraso

---

## Contexto del caso de negocio

| | |
|---|---|
| **Empresa** | empresa — área de delivery y gestión de proyectos software |
| **Problema de negocio** | Detectar en las primeras semanas de un proyecto si existe riesgo de entrega tardía, para reasignar recursos o renegociar plazos antes de que el retraso sea inevitable |
| **Datos disponibles** | 200 proyectos históricos con variables de planificación: tamaño del equipo, horas estimadas vs. imputadas, número de cambios de requisitos, experiencia del equipo, complejidad técnica y variable objetivo retraso_proyecto |
| **Técnica aplicada** | Clasificación supervisada con RandomForestClassifier con class_weight=balanced para manejar el desbalance; análisis de umbral (0.5 vs. 0.3) para ajustar la sensibilidad según el coste del error |
| **Salida del modelo** | Probabilidad de retraso por proyecto (0-1) y clasificación binaria; curva ROC y AUC para evaluar la capacidad discriminativa del modelo |
| **Valor operativo** | Permite al director de delivery priorizar la supervisión en los proyectos de mayor riesgo y tomar decisiones de refuerzo de equipo o ajuste de alcance con semanas de antelación |

In [ ]:
import os, sys
from pathlib import Path

# Configuracion de entorno: ajusta CWD y descarga datos segun el entorno de ejecucion
_BASE_URL = "https://raw.githubusercontent.com/amador2001/ia-datos/main/"
_CSVS = ["proyectos_software.csv"]

if "google.colab" in sys.modules:
    import urllib.request
    Path("datos").mkdir(exist_ok=True)
    for _csv in _CSVS:
        _dest = Path("datos") / _csv
        if not _dest.exists():
            urllib.request.urlretrieve(_BASE_URL + _csv, str(_dest))
            print(f"Descargado: {_csv}")
elif "__vsc_ipynb_file__" in dir():
    os.chdir(Path(__vsc_ipynb_file__).parent)
elif not Path("datos").exists():
    for _p in [Path("Jupyter_notebooks"), Path("../Jupyter_notebooks")]:
        if (_p / "datos").exists():
            os.chdir(_p)
            break

print(f"Entorno listo. CWD: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# Cargar datos de proyectos software
df = pd.read_csv("datos/proyectos_software.csv")

print("Primeras filas del dataset:")
display(df.head(10))

print("\nEstadisticas descriptivas:")
display(df.describe().round(2))

df.info()

print(f"\nTasa de proyectos con retraso: {df['retraso_proyecto'].mean():.1%}")
print(f"Dias de retraso (cuando hay retraso): media={df[df['dias_retraso']>0]['dias_retraso'].mean():.0f} dias")

### Variables del dataset

| Variable | Descripcion | Relacion con retraso |
|---|---|---|
| tickets_abiertos | Total de tickets en el backlog | Alta: acumulacion indica deuda tecnica |
| tickets_criticos | Tickets bloqueantes | Muy alta: bloquean el avance |
| cambios_scope_mes | Cambios de requisitos en el mes | Muy alta: scope creep es la causa mas comun |
| commits_semana | Actividad de desarrollo media | Media inversa: baja actividad puede indicar bloqueo |
| prs_abiertas | Pull requests sin revisar | Media: revision lenta frena el avance |
| tamano_equipo | Numero de desarrolladores | Media: equipos muy grandes tienen mas coordinacion |
| rotacion_equipo | % de cambios en el equipo | Alta: la rotacion destruye contexto |
| horas_estimadas | Estimacion inicial del proyecto | Referencia para calcular desviacion |
| horas_reales_acumuladas | Horas consumidas hasta ahora | Alta desviacion sobre estimacion = riesgo |
| num_dependencias_externas | APIs, equipos externos | Media: cada dependencia es un punto de fallo |
| incidencias_cliente | Problemas reportados por el cliente | Alta: indica insatisfaccion o entregables defectuosos |

> **Variable derivada clave**: ratio = horas_reales / horas_estimadas
> Si ratio > 1 el proyecto ya ha superado la estimacion.

In [ ]:
# Anadir columna ratio de horas para el analisis
df["ratio_horas"] = df["horas_reales_acumuladas"] / df["horas_estimadas"]

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
axes = axes.flatten()

features_plot = ["tickets_abiertos","tickets_criticos","cambios_scope_mes",
                 "commits_semana","prs_abiertas","tamano_equipo",
                 "rotacion_equipo","ratio_horas",
                 "num_dependencias_externas","incidencias_cliente"]

for i, feat in enumerate(features_plot):
    for label, color in [(0,"steelblue"),(1,"tomato")]:
        df[df["retraso_proyecto"]==label][feat].plot.kde(
            ax=axes[i], label=f"retraso={label}", color=color, alpha=0.7)
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)

# Correlacion con target
corr = df[features_plot + ["retraso_proyecto"]].corr()["retraso_proyecto"].drop("retraso_proyecto").sort_values()
corr.plot(kind="barh", ax=axes[10], color=["tomato" if v > 0 else "steelblue" for v in corr])
axes[10].set_title("Correlacion con retraso_proyecto")
axes[10].axvline(0, color="black", linewidth=0.5)

axes[11].axis("off")
plt.suptitle("Distribucion de features: proyectos en plazo vs con retraso", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Anadir ratio_horas como feature
FEATURES = ["tickets_abiertos","tickets_criticos","cambios_scope_mes",
            "commits_semana","prs_abiertas","tamano_equipo",
            "rotacion_equipo","ratio_horas",
            "num_dependencias_externas","incidencias_cliente"]
TARGET = "retraso_proyecto"

X = df[FEATURES]
y = df[TARGET]

# Split aleatorio (los proyectos son independientes, no hay serie temporal)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Filas entrenamiento: {len(X_train)}")
print(f"Filas test:          {len(X_test)}")

# Normalizar con StandardScaler
# RandomForest no lo necesita, pero conviene tener el dataset normalizado disponible
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\nDataset de entrenamiento normalizado (primeras 3 filas):")
display(pd.DataFrame(X_train_scaled, columns=FEATURES).head(3).round(3))

In [ ]:
# Entrenar RandomForest con pesos balanceados
# class_weight="balanced" hace que el modelo penalice mas los errores
# en proyectos con retraso (clase minoritaria)
rf = RandomForestClassifier(
    n_estimators=200,       # 200 arboles para mayor estabilidad
    random_state=42,
    class_weight="balanced" # compensar el desbalance de clases
)
rf.fit(X_train, y_train)

# Predicciones sobre test
y_pred = rf.predict(X_test)           # clase predicha (0/1) con umbral 0.5
y_prob = rf.predict_proba(X_test)[:,1] # probabilidad de retraso

print("Resultados con umbral 0.5:")
print(classification_report(y_test, y_pred, target_names=["En plazo","Con retraso"]))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.3f}")

# Mostrar tabla con probabilidades de riesgo
tabla = pd.DataFrame({
    "retraso_real":  y_test.values[:10],
    "prob_retraso":  y_prob[:10].round(3),
    "pred_retraso":  y_pred[:10]
})
print("\nPrimeros 10 proyectos del test:")
display(tabla)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Feature importance
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fi.plot(kind="barh", ax=axes[0,0], color="steelblue")
axes[0,0].set_title("Importancia de variables (Random Forest)")
axes[0,0].set_xlabel("Importancia relativa")

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[0,1].plot(fpr, tpr, color="tomato", label=f"RF (AUC={auc:.3f})")
axes[0,1].plot([0,1],[0,1],"k--")
axes[0,1].set_title("Curva ROC")
axes[0,1].set_xlabel("Tasa Falsos Positivos")
axes[0,1].set_ylabel("Tasa Verdaderos Positivos")
axes[0,1].legend()

# Confusion matrix umbral 0.5
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=["Pred:Plazo","Pred:Retraso"],
            yticklabels=["Real:Plazo","Real:Retraso"], ax=axes[1,0])
axes[1,0].set_title("Matriz de confusion (umbral=0.5)")

# Confusion matrix umbral 0.3 (mas sensible al retraso)
umbral_bajo = 0.3
y_pred_bajo = (y_prob >= umbral_bajo).astype(int)
cm_bajo = confusion_matrix(y_test, y_pred_bajo)
sns.heatmap(cm_bajo, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Pred:Plazo","Pred:Retraso"],
            yticklabels=["Real:Plazo","Real:Retraso"], ax=axes[1,1])
axes[1,1].set_title(f"Matriz de confusion (umbral={umbral_bajo})\nMas sensible: detecta mas retrasos, mas falsas alarmas")

plt.tight_layout()
plt.show()

In [ ]:
# Prediccion para un proyecto especifico
proyecto_nuevo = pd.DataFrame([{
    "tickets_abiertos":         45,    # backlog elevado
    "tickets_criticos":          8,    # varios bloqueantes
    "cambios_scope_mes":         6,    # muchos cambios de requisitos
    "commits_semana":           12.0,  # actividad moderada
    "prs_abiertas":             18,    # muchas PRs sin revisar
    "tamano_equipo":             7,
    "rotacion_equipo":          0.28,  # alta rotacion (28% del equipo cambio)
    "ratio_horas":              1.35,  # 35% por encima de la estimacion
    "num_dependencias_externas": 5,
    "incidencias_cliente":       9
}])

prob_retraso = rf.predict_proba(proyecto_nuevo)[0][1]
prediccion   = rf.predict(proyecto_nuevo)[0]

print(f"Probabilidad de retraso: {prob_retraso:.1%}")
print(f"Prediccion: {'RETRASO ESPERADO' if prediccion == 1 else 'EN PLAZO'}")
print(f"\nSenales de riesgo detectadas:")
print(f"  - ratio_horas > 1.0:        el proyecto ya supero la estimacion")
print(f"  - cambios_scope_mes = 6:    scope creep activo")
print(f"  - tickets_criticos = 8:     varios bloqueantes sin resolver")
print(f"  - rotacion_equipo = 28%:    perdida de contexto significativa")

### Cuando NO usar este modelo

| Situacion | Por que no funciona |
|---|---|
| Proyecto nuevo sin datos operativos | El modelo necesita metricas reales (tickets, horas) para funcionar |
| Metodologia muy diferente al historico | Si los proyectos entrenados eran Waterfall y el nuevo es SAFe, los patrones cambian |
| Retrasos causados por factores externos | Cambios legales, problemas de proveedor: el modelo no los ve |
| Como unico criterio de decision | El modelo es una senal de alerta, no una sentencia: siempre validar con el equipo |

> **Regla del umbral**: si el coste de un retraso no detectado (falso negativo)
> es mayor que el coste de una alerta innecesaria (falso positivo),
> usa un umbral bajo (0.3). Si el equipo tiene capacidad limitada de atencion,
> sube el umbral para reducir las falsas alarmas.